# Trabajo final Python para Ciencia de Datos
### Santiago Macedo, Dylan Gandaria 

In [2]:
# Librerías
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
SEED = 42

## 1 - Introducción
Para el trabajo elegimos un conjunto de datos que contiene registros que exploran la relacion que existe entre las horas de sueño y de uso de pantallas con el rendimiento academico de estudiantes. 
En este conjunto podremos ver informacion demografica, patrones de sueño, habitos de uso de pantallas, resultados academicos y de estilo de vida.
Este conjunto de datos esta construido para explorar especificamente el vinculo que existe entre el tiempo de sueño y el uso de pantallas con los resultados academicos, debido a que estos son considerados grandes factores de riesgo academico.
Es un conjunto de datos simulado artificialmente y no extraido de estudiantes reales, con el objetivo de que las horas de sueño, estudio y pantalla predigan de manera significativa los resultados academicos de los estudiantes. Contiene outliers introducidos de forma externa al algoritmo de simulacion utilizado. El proposito de este conjunto de datos es educacional y esta hecho para practicar, por lo que **no puede ser utilizado para extraer conclusiones sobre ningun fenomeno observado en el mundo real.**


In [3]:
data = pd.read_csv('../data/raw/data.csv')

## 2 - Datos
Este conjunto proviene de la plataforma Kaggle y se puede ver en el siguiente [link](https://www.kaggle.com/datasets/aamir28/student-sleep-screen-time-and-academic-performance?resource=download)

El conjunto contiene 500 observaciones de estudiantes simulados.
| Variable | Tipo | Descripción |
|---|---|---|
| `student_id` | Identificador | Código único del estudiante |
| `age` | Numérica (discreta) | Edad en años (17 a 27). |
| `gender` | Categórica | Género: Female, Male, Non-binary, Prefer not to say. Presenta categorías duplicadas por escritura (`male`, `F`, `M`, etc.)  |
| `year_of_study` | Ordinal | Año de la carrera que cursa (1 a 5). |
| `major_category` | Categórica | Área de estudio: STEM, Business, Social Sciences, Health Sciences, Humanities, Arts, Law. |
| `residence_type` | Categórica | Tipo de vivienda: Dormitory, Off-campus Apartment, With Family, Homestay. |
| `chronotype` | Categórica | Cronotipo: Early Bird, Intermediate, Night Owl. |
| `avg_bedtime_hour` | Numérica (continua) | Hora promedio de acostarse, en formato 0 a 24 (por ejemplo, 23.1 = 23:06; 0.9 = 00:54). |
| `avg_sleep_hours_weekday` | Numérica (continua) | Horas promedio de sueño en días de semana. |
| `weekend_sleep_delta_hours` | Numérica (continua) | Diferencia en horas de sueño entre fin de semana y días de semana (valores positivos indican que duerme más el fin de semana). |
| `pre_sleep_screen_time_min` | Numérica (continua) | Minutos de uso de pantallas antes de dormir. |
| `daily_screen_time_hours` | Numérica (continua) | Horas diarias totales de uso de pantallas. |
| `primary_device` | Categórica | Dispositivo principal: Smartphone, Laptop, Tablet. |
| `device_free_before_bed` | Categórica (binaria) | Si evita dispositivos antes de dormir (Yes/No). |
| `caffeine_drinks_per_day` | Numérica (discreta) | Bebidas con cafeína por día. |
| `naps_per_week` | Numérica (discreta) | Siestas por semana (0 a 7). |
| `sleep_quality_score` | Ordinal | Calidad del sueño autopercibida (escala de 1 a 10). |
| `exercise_days_per_week` | Numérica (discreta) | Días de ejercicio por semana (0 a 7). |
| `energy_level` | Ordinal | Nivel de energía autopercibido (escala de 1 a 10). |
| `concentration_difficulty` | Ordinal | Dificultad de concentración autopercibida (escala de 1 a 10). |
| `weekly_study_hours` | Numérica (continua) | Horas de estudio por semana. |
| `class_attendance_percent` | Numérica (continua) | Porcentaje de asistencia a clase. |
| `has_part_time_job` | Categórica (binaria) | Si tiene trabajo de medio tiempo (Yes/No). |
| `semester_type` | Categórica | Semestre en que se registró: Fall, Spring, Summer. |
| `cgpa` | Numérica (continua) | Promedio acumulado de calificaciones (rango observado: 1.89 a 3.95). **Variable de interés.** |

### Aspectos relevantes del conjunto

- **Valores faltantes:** `energy_level` (15), `concentration_difficulty` (14), `device_free_before_bed` (9), `cgpa` (7) y `sleep_quality_score` (6).
- **Inconsistencias de formato:** la variable `gender` mezcla mayúsculas, minúsculas y abreviaturas.
- **Valores extremos:** se observan casos atípicos en horas de estudio, uso de pantalla y horas de sueño que fueron introducidos a proposito para simular respuestas reales de encuestas.

In [ ]:
data.groupby('energy_level', 'concentration_difficulty', 'device_free')

,student_id,age,gender,year_of_study,major_category,residence_type,chronotype,avg_bedtime_hour,avg_sleep_hours_weekday,weekend_sleep_delta_hours,pre_sleep_screen_time_min,daily_screen_time_hours,primary_device,device_free_before_bed,caffeine_drinks_per_day,naps_per_week,sleep_quality_score,exercise_days_per_week,energy_level,concentration_difficulty,weekly_study_hours,class_attendance_percent,has_part_time_job,semester_type,cgpa
0,STU2001,21,Male,3,Humanities,With Family,Early Bird,22.3,7.6,2.0,12,7.1,Tablet,No,2,0,10.0,7,5.0,7.0,15.6,80.0,No,Spring,2.99
1,STU2002,20,Female,2,STEM,Dormitory,Intermediate,23.1,5.9,2.7,49,5.7,Laptop,Yes,1,3,7.0,3,6.0,7.0,17.5,78.9,No,Summer,2.88
2,STU2003,20,Male,4,Social Sciences,With Family,Intermediate,23.4,8.4,1.8,22,2.3,Smartphone,No,2,1,9.0,2,5.0,1.0,19.0,69.7,Yes,Spring,3.15
3,STU2004,22,F,1,Humanities,Off-campus Apartment,Night Owl,0.9,5.1,2.4,32,4.8,Laptop,No,1,3,6.0,4,3.0,7.0,7.3,59.1,No,Fall,2.27
4,STU2005,23,Female,4,Social Sciences,Off-campus Apartment,Early Bird,23.4,5.9,2.0,27,4.5,Smartphone,No,4,2,7.0,6,8.0,3.0,23.6,79.9,No,Summer,2.74


# 3 - Analisis exploratorio